## Sample Statistics script

This is script is used to calculate the statistics in results section 3.1.1

In [ ]:
# Import packages
import pandas as pd
import numpy as np

# read data
sample_information = pd.read_csv("data/sample_information.csv")


In [ ]:
# Function to calculate population statistics

def process_general_data(df: pd.DataFrame) -> dict:

    # Create extra columns for simplified analysis
    df["sample_n_control"] = df["sample_n_total"] - df["sample_n_adhd"]

    df["sample_adhd_female_n"] = (df["sample_n_adhd"] * df["sample_adhd_female"]).round()
    df["sample_control_female_n"] = (df["sample_n_control"] * df["sample_control_female"]).round()

    df["sample_total_female_n"] = (df["sample_adhd_female_n"] + df["sample_control_female_n"])
    df["sample_total_female"] = (df["sample_total_female_n"] / df["sample_n_total"])

    # Extract important variabels
    n1, n2 = df["sample_n_adhd"], df["sample_n_control"]
    m1, m2 = df["sample_adhd_age_mean"], df["sample_control_age_mean"]
    s1, s2 = df["sample_adhd_age_sd"], df["sample_control_age_sd"]

    # Account for empty rows
    valid_smd = (
            (n1 > 1)
            & (n2 > 1)
            & (~m1.isna())
            & (~m2.isna())
            & (~s1.isna())
            & (~s2.isna())
        )
    
    df["age_smd_adhd_vs_control"] = np.nan
    s_pooled = np.sqrt(
        (
            (n1[valid_smd] - 1) * (s1[valid_smd] ** 2)
            + (n2[valid_smd] - 1) * (s2[valid_smd] ** 2)
        )
        / (n1[valid_smd] + n2[valid_smd] - 2)
    )
    df.loc[valid_smd, "age_smd_adhd_vs_control"] = (
        m1[valid_smd] - m2[valid_smd]
    ) / s_pooled

    # 4. Synthesized Pooled Statistics
    def calc_pooled_age(n_series, m_series, sd_series):
        valid = (~n_series.isna()) & (~m_series.isna()) & (~sd_series.isna())
        n, m, s = n_series[valid], m_series[valid], sd_series[valid]
        tot_n = n.sum()
        p_mean = (n * m).sum() / tot_n
        var_w = ((n - 1) * (s**2)).sum()
        var_b = (n * ((m - p_mean) ** 2)).sum()
        p_sd = np.sqrt((var_w + var_b) / (tot_n - 1))
        return p_mean, p_sd

    adhd_mean, adhd_sd = calc_pooled_age(
        df["sample_n_adhd"],
        df["sample_adhd_age_mean"],
        df["sample_adhd_age_sd"],
    )
    ctrl_mean, ctrl_sd = calc_pooled_age(
        df["sample_n_control"],
        df["sample_control_age_mean"],
        df["sample_control_age_sd"],
    )



    summary_stats = {
            "Total Included Studies (K)": int(df["study_id"].nunique()),
            "Total Sample Size (N)": int(df["sample_n_total"].sum()),
            "ADHD Cohort (N)": int(df["sample_n_adhd"].sum()),
            "Control Cohort (N)": int(df["sample_n_control"].sum()),
            "Study Size Median (IQR)": f"{df['sample_n_total'].median():.1f} [{df['sample_n_total'].quantile(0.25):.1f} – {df['sample_n_total'].quantile(0.75):.1f}]",
            "Pooled ADHD Age (Mean ± SD)": f"{adhd_mean:.2f} ± {adhd_sd:.2f} years",
            "Pooled Control Age (Mean ± SD)": f"{ctrl_mean:.2f} ± {ctrl_sd:.2f} years",
            "ADHD % Female": f"{(df['sample_adhd_female_n'].sum() / df['sample_n_adhd'].dropna().sum()) * 100:.1f}%",
            "Control % Female": f"{(df['sample_control_female_n'].sum() / df['sample_n_control'].dropna().sum()) * 100:.1f}%",
        }

    return summary_stats

In [ ]:
print(process_general_data(sample_information))

In [ ]:
# Load the dataset
df = pd.read_csv('clean_age_medication.csv')

target_columns = [
    'Age: Group-level Matching (Bool)',
    'Age: Statistical Control in Analysis (Bool)',
    'Stimulant Handling (Bool)',
    'No other Medications Used in Clinical Group (Bool)',
    'Medication: Statistical Control & Analysis (Bool)'
]

print("--- Counts for 1, 0, and NaN ---")

for col in target_columns:
    # Check if the column actually exists in the file to avoid errors
    if col in df.columns:
        # Get counts of all values, including NaNs
        counts = df[col].value_counts(dropna=False)
        
        # Safely extract the counts (defaults to 0 if the value isn't present)
        count_1 = counts.get(1.0, counts.get(1, 0))
        count_0 = counts.get(0.0, counts.get(0, 0))
        
        # NaNs can be tricky to extract from value_counts directly, 
        # so we can also use isna().sum() for a foolproof NaN count
        count_nan = df[col].isna().sum()
        
        print(f"\n{col}:")
        print(f"  Positives (1): {count_1}")
        print(f"  Negatives (0): {count_0}")
        print(f"  Missing (NaN): {count_nan}")
    else:
        print(f"\n{col}: Column not found in the dataset.")

# 2. Extract and count washout types from 'Stimulant Handling'
print("\n--- Stimulant Handling: Washout Counts ---")

def parse_washout(text):
    text = str(text).lower()
    
    # Check for specific timeframes
    if '12 to 24-hour' in text:
        return '12-24h washout'
    elif '24' in text:
        return '24h washout'
    elif '48' in text:
        return '48h washout'
    elif '12' in text:
        return '12h washout'
    elif '3 weeks' in text:
        return '3 weeks washout'
    elif 'withheld on test days' in text or 'medication-free on the day' in text:
        return 'Day-of testing washout'
    elif 'standard washout' in text:
        return 'Standard washout (unspecified)'
    
    # Check for unmedicated/active statuses
    elif 'naive' in text or 'naïve' in text:
        return 'Treatment-naive'
    elif 'unmedicated' in text or 'exclusion' in text or 'medication-free' in text:
        return 'Unmedicated / Excluded'
    elif 'active' in text or 'regular dose' in text:
        return 'Active (On-Meds)'
        
    return 'Other'

# Apply the function and count the occurrences
df['Washout_Period'] = df['Stimulant Handling'].apply(parse_washout)
washout_counts = df['Washout_Period'].value_counts()
print(washout_counts.to_string())